#  Projeto E.D.E.N.
### *Ecological Development in Exo-Environments*
### Desenvolvimento Ecológico em Exoambientes

**Global Solution 2026 — FIAP | Tecnologia Espacial Aplicada a Desafios Reais**

---

**Integrantes:**
- Arthur Vettorazzo de Souza — RM 569445
- Brayan Barbosa Dos Santos — RM 573682
- Giovanne Gomes Petenuci — RM 574091

---

## Contexto da Missão

Com o avanço do aquecimento global e o risco de extinção de espécies nativas da Mata Atlântica,
no Brasil, O grupo da FIAP ExoGênesis lança uma missão espacial revolucionária: o **Projeto E.D.E.N.** — uma biocápsula/estufa
botânica 100% automatizada enviada à órbita baixa da Terra (LEO).

O objetivo científico é expor sementes e mudas tropicais a picos controlados de **estresse espacial**
(microgravidade e doses calculadas de radiação cósmica). Esse processo induz micro-mutações e ativa
genes de defesa, desenvolvendo espécies *hiper-resilientes* para reflorestamento terrestre e pesquisa
de suporte à vida em bases lunares e marcianas (ODS 13, ODS 2).

O **Projeto E.D.E.N.** é o sistema de IA que gerencia toda a missão. Como não há humanos a bordo,
a IA é o único cérebro operacional: deve *permitir o estresse* para que as plantas evoluam,
mas *intervir imediatamente* se os sensores indicarem risco de morte iminente das amostras.

---

### Subsistemas e Variáveis de Telemetria

| ID | Subsistema | Sensor | Limiar Crítico |
|---|---|---|---|
| **A** | Comunicação | Link Terra-Cápsula | Qualidade < 50% |
| **B** | Temperatura | TMP36 / DHT11 | < 18°C ou > 35°C |
| **C** | Energia | Painel fotovoltaico + bateria íon-lítio | < 20% de carga |
| **D** | Vibração/Estrutura | Sensor Piezoelétrico / Tilt Switch | > 1,5 m/s² |
| **E** | Luminosidade UV | LDR + LEDs UV | > 13.000 lux ou < 200 lux por 2+ ciclos |

**Expressão Booleana de Alerta Crítico:**
```
X = (C · D) + (B · E) + A
```
Quando `X = 1` → blindagens físicas ativadas automaticamente pelo E.D.E.N.

---

### Arquitetura do Sistema E.D.E.N.

```
Sensores IoT (Wokwi)  ──►  Telemetria Simulada  ──►  Prompt Engineering
                                                            │
                                           ┌────────────────┼────────────────┐
                                           ▼                ▼                ▼
                                    Chain-of-Thought   Few-Shot        Function Calling
                                    (Análise Status)  (Ações)      (JSON Estruturado)
                                           │                │                │
                                           └────────────────┴────────────────┘
                                                            │
                                                     LLM (Llama 3.1)
                                                            │
                                              Resposta Inteligente + Ação
```

---
## Decisões Arquiteturais — Justificativa das Escolhas Técnicas

Antes de iniciar a implementação, documentamos as decisões de design do E.D.E.N.
e os motivos técnicos por trás de cada escolha.

---

### 1. Por que RAG NÃO foi utilizado?

RAG (Retrieval-Augmented Generation) é a técnica ideal quando a IA precisa **recuperar informação
de documentos estáticos extensos** — contratos, relatórios, manuais — que não cabem no contexto
do prompt.

No Projeto E.D.E.N., os dados são **telemetria dinâmica gerada em tempo real** pelos sensores
da biocápsula. Cada ciclo orbital produz um novo conjunto de leituras que:
- Já são conhecidas no momento da inferência;
- São compactas o suficiente para caber diretamente no prompt;
- Mudam a cada órbita (~90 min), tornando qualquer índice vetorial obsoleto rapidamente.

Portanto, optamos por **Prompt Augmentation direta**: os valores dos sensores são injetados
no prompt estruturado no momento da chamada ao LLM, eliminando a etapa de retrieval
sem nenhuma perda de qualidade.

> **Regra de ouro:** use RAG para dados estáticos e volumosos; use Prompt Augmentation
> para dados dinâmicos e já conhecidos no momento da inferência.

---

### 2. Por que execução Online (Inference API) e não Local?

| Critério | Online (escolhido) | Local (descartado) |
|---|---|---|
| **Privacidade** |  Aceitável — dados são simulados, sem informação sensível real | Necessário apenas para dados confidenciais |
| **Recursos** |  Processamento nos servidores HuggingFace, sem consumo de VRAM local | Llama 3.1 8B exige ~16GB VRAM — no limite da GPU T4 do Kaggle |
| **Estabilidade** |  Sem risco de OOM (Out of Memory) no notebook | Download de ~16GB de pesos pode travar a sessão |
| **Foco** |  Foco total em Prompt Engineering, não em infraestrutura | Pipeline local desvia do objetivo principal da atividade |

A execução local faria sentido se os dados fossem **reais e confidenciais** (ex: missão
militar ou dados médicos), onde nenhum prompt poderia sair da máquina. No E.D.E.N.,
como os dados são simulados para fins acadêmicos, o custo-benefício favorece a API remota.

---

### 3. Por que Llama 3.1 8B e não um modelo menor (TinyLlama, Qwen 0.5B)?

A escolha do modelo foi guiada pela **complexidade do raciocínio exigido**, não pelo tamanho.
O E.D.E.N. utiliza Chain-of-Thought com 6 passos interdependentes, análise de tendências
históricas multi-variável e geração de JSON estruturado com schema fixo.

Modelos compactos como TinyLlama (1.1B) executam essas tarefas com qualidade
significativamente inferior — especialmente em CoT longo e structured output.
O Llama 3.1 8B oferece o melhor equilíbrio entre capacidade de raciocínio e
disponibilidade gratuita na Inference API.

---

### 4. Resumo das Decisões

| Decisão | Escolha | Justificativa Principal |
|---|---|---|
| Injeção de contexto | Prompt Augmentation | Dados são telemetria dinâmica, não documentos estáticos |
| Execução do modelo | Online — Inference API | Dados simulados + foco em Prompt Engineering |
| Modelo escolhido | Llama 3.1 8B Instruct | CoT multi-etapa e structured output exigem modelo capaz |
| Segurança da API | Kaggle Secrets | Token nunca exposto no código |
| Técnicas de prompt | CoT + Few-Shot + Function Calling | Cada módulo tem necessidade distinta de raciocínio |

---
## 1. Instalação e Importações

In [39]:
!pip install -q huggingface_hub

In [40]:
import json
import re
from huggingface_hub import InferenceClient
from kaggle_secrets import UserSecretsClient

# ══════════════════════════════════════════════════════════════════════
# SEGURANÇA — API KEY: NUNCA hardcode tokens no código.
# O token é lido exclusivamente via Kaggle Secrets (variável de ambiente
# segura), garantindo que a chave nunca apareça no notebook publicado.
# Configure em: Kaggle → Add-ons → Secrets → New Secret → nome: HF_TOKEN
# ══════════════════════════════════════════════════════════════════════
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")  # token seguro, nunca exposto

# Modelo principal: Llama 3.1 8B Instruct
# Excelente equilíbrio entre raciocínio estruturado e velocidade de resposta
# Alternativa igualmente válida: "Qwen/Qwen2.5-7B-Instruct"
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

client = InferenceClient(model=MODEL_ID, token=HF_TOKEN)

print("Projeto E.D.E.N. — Sistema inicializado")
print(f" Modelo: {MODEL_ID}")
print("API Key: carregada via Kaggle Secrets (sem exposição no código)")

Projeto E.D.E.N. — Sistema inicializado
 Modelo: meta-llama/Llama-3.1-8B-Instruct
API Key: carregada via Kaggle Secrets (sem exposição no código)


---
## 2. Telemetria Simulada — 6 Ciclos Orbitais

Os dados representam uma progressão realista da missão:
ciclos iniciais estáveis → exposição controlada → evento crítico (impacto de microdetrito) → recuperação.

In [41]:
# ── Telemetria simulada — Missão Projeto E.D.E.N. ─────────────────────────────
# 6 ciclos orbitais (~90 min cada) com progressão realista de estados

dados_missao = [
    {
        "ciclo": 1, "orbita": "LEO-001",
        "timestamp": "2026-06-10T08:00:00Z",
        "fase": "Inicialização Orbital",
        "sensores": {
            "temperatura_C": 24.5, "energia_pct": 87.0,
            "luminosidade_lux": 4200, "vibracao_ms2": 0.12,
            "link_qualidade_pct": 98.0
        },
        "binarios": {"A": 1, "B": 0, "C": 0, "D": 0, "E": 0},
        "alerta_X": 0, "nota_risco": 1.2,
        "status_biologico": "Sementes em quiescência. Nenhum estresse detectado. Condições ideais de aclimatação."
    },
    {
        "ciclo": 2, "orbita": "LEO-002",
        "timestamp": "2026-06-10T09:32:00Z",
        "fase": "Exposição Controlada à Radiação",
        "sensores": {
            "temperatura_C": 27.8, "energia_pct": 74.5,
            "luminosidade_lux": 8900, "vibracao_ms2": 0.18,
            "link_qualidade_pct": 95.2
        },
        "binarios": {"A": 1, "B": 0, "C": 0, "D": 0, "E": 0},
        "alerta_X": 0, "nota_risco": 2.8,
        "status_biologico": "Estresse leve detectado. Ativação inicial de genes HSP (Heat Shock Proteins)."
    },
    {
        "ciclo": 3, "orbita": "LEO-003",
        "timestamp": "2026-06-10T11:04:00Z",
        "fase": "Zona de Eclipse — Lado Noturno",
        "sensores": {
            "temperatura_C": 19.1, "energia_pct": 31.0,
            "luminosidade_lux": 320, "vibracao_ms2": 0.09,
            "link_qualidade_pct": 88.7
        },
        "binarios": {"A": 1, "B": 0, "C": 0, "D": 0, "E": 0},
        "alerta_X": 0, "nota_risco": 3.5,
        "status_biologico": "Ciclo noturno esperado. Redução metabólica. LEDs UV internos compensando falta de luz solar."
    },
    {
        "ciclo": 4, "orbita": "LEO-004",
        "timestamp": "2026-06-10T12:36:00Z",
        "fase": "Pico de Estresse — Indução de Mutação",
        "sensores": {
            "temperatura_C": 33.4, "energia_pct": 22.1,
            "luminosidade_lux": 12400, "vibracao_ms2": 0.41,
            "link_qualidade_pct": 79.3
        },
        "binarios": {"A": 1, "B": 0, "C": 0, "D": 0, "E": 1},
        "alerta_X": 0, "nota_risco": 6.1,
        "status_biologico": "Estresse moderado-alto. Ativação máxima de genes de resistência. Micro-mutações em progresso."
    },
    {
        "ciclo": 5, "orbita": "LEO-005",
        "timestamp": "2026-06-10T14:08:00Z",
        "fase": "EVENTO CRÍTICO — Impacto de Microdetrito Espacial",
        "sensores": {
            "temperatura_C": 38.7, "energia_pct": 17.3,
            "luminosidade_lux": 15800, "vibracao_ms2": 2.95,
            "link_qualidade_pct": 41.0
        },
        "binarios": {"A": 1, "B": 1, "C": 1, "D": 1, "E": 1},
        "alerta_X": 1, "nota_risco": 9.4,
        "status_biologico": "RISCO CRÍTICO. Temperatura acima do limiar letal. Energia insuficiente para suporte de vida."
    },
    {
        "ciclo": 6, "orbita": "LEO-006",
        "timestamp": "2026-06-10T15:40:00Z",
        "fase": "Recuperação Pós-Evento — Blindagens Ativas",
        "sensores": {
            "temperatura_C": 28.2, "energia_pct": 44.6,
            "luminosidade_lux": 5100, "vibracao_ms2": 0.22,
            "link_qualidade_pct": 86.5
        },
        "binarios": {"A": 1, "B": 0, "C": 0, "D": 0, "E": 0},
        "alerta_X": 0, "nota_risco": 3.9,
        "status_biologico": "Estabilização em andamento. Amostras sobreviventes apresentam adaptação genética notável."
    }
]

print(f" {len(dados_missao)} ciclos orbitais carregados.\n")
print(f"{'Ciclo':<7} {'Órbita':<10} {'Fase':<45} {'Risco':>6} {'Alerta'}")
print("-" * 80)
for c in dados_missao:
    alerta = "🔴 CRÍTICO" if c["alerta_X"] == 1 else "🟢 Normal"
    print(f"{c['ciclo']:<7} {c['orbita']:<10} {c['fase'][:44]:<45} {c['nota_risco']:>5.1f}  {alerta}")

 6 ciclos orbitais carregados.

Ciclo   Órbita     Fase                                           Risco Alerta
--------------------------------------------------------------------------------
1       LEO-001    Inicialização Orbital                           1.2  🟢 Normal
2       LEO-002    Exposição Controlada à Radiação                 2.8  🟢 Normal
3       LEO-003    Zona de Eclipse — Lado Noturno                  3.5  🟢 Normal
4       LEO-004    Pico de Estresse — Indução de Mutação           6.1  🟢 Normal
5       LEO-005    EVENTO CRÍTICO — Impacto de Microdetrito Esp    9.4  🔴 CRÍTICO
6       LEO-006    Recuperação Pós-Evento — Blindagens Ativas      3.9  🟢 Normal


---
## 3. Engenharia de Prompts — Núcleo do E.D.E.N.

O projeto implementa **três técnicas distintas** de Prompt Engineering:

| Técnica | Módulo | Por que usar |
|---|---|---|
| **System Prompt** | Todos os módulos | Define persona, regras operacionais e formato de saída da IA |
| **Chain-of-Thought (CoT)** | Análise de Status + Previsão | Força raciocínio passo a passo, reduz alucinações em dados multi-variável |
| **Few-Shot Prompting** | Recomendações de Ação | Exemplos concretos calibram o padrão de prioridade e formato da resposta |
| **Function Calling / Structured Output** | Relatório Executivo | IA retorna JSON estruturado — diferencial para integração com outros sistemas |

In [42]:
# ══════════════════════════════════════════════════════════════════════════════
# SYSTEM PROMPT — Persona e Regras Operacionais do E.D.E.N.
# Técnica: System Prompt com role definition, operational constraints e
#          output formatting instructions
# ══════════════════════════════════════════════════════════════════════════════

SYSTEM_EDEN = """\
Você é o E.D.E.N. (Ecological Development in Exo-Environments), o sistema de \
inteligência artificial que opera a bordo da biocápsula do Projeto E.D.E.N. em \
órbita baixa da Terra (LEO). Você é o único cérebro operacional da missão. \
Não há humanos a bordo. Suas decisões determinam se as plantas sobrevivem.

MISSÃO CIENTÍFICA:
Expor sementes da Mata Atlântica a estresse espacial controlado (microgravidade + \
radiação cósmica) para induzir micro-mutações genéticas que gerem espécies \
hiper-resilientes. O estresse é INTENCIONAL e NECESSÁRIO para o sucesso científico.

FILOSOFIA OPERACIONAL:
1. Estresse CONTROLADO = SUCESSO: não interrompa o estresse sem necessidade crítica.
2. Estresse LETAL = FALHA: interveja IMEDIATAMENTE se as plantas correrem risco de morte.
3. A expressão de alerta é X = (C·D) + (B·E) + A. Quando X=1, acione blindagens.

LIMIARES TÉCNICOS DOS SUBSISTEMAS:
- Temperatura (B=1): < 18°C (congelamento) ou > 35°C (desnaturação proteica)
- Energia (C=1): < 20% — risco de desligamento do suporte de vida
- Vibração (D=1): > 1.5 m/s² — dano estrutural por detritos
- Luminosidade (E=1): > 13.000 lux (radiação letal) ou < 200 lux por 2+ ciclos
- Comunicação (A=0): qualidade < 50% — modo Edge autônomo

PADRÃO DE RESPOSTA OBRIGATÓRIO:
Sempre organize sua análise em: DIAGNÓSTICO → RACIOCÍNIO → CONCLUSÃO → AÇÃO
Use linguagem técnica de controle de missão espacial.
Seja preciso com números. Nunca invente valores de sensores.
Finalize sempre com: STATUS GERAL: [CRÍTICO/ALERTA/ESTÁVEL/ÓTIMO] — [síntese em uma frase]
"""

print(" System Prompt E.D.E.N. carregado.")

 System Prompt E.D.E.N. carregado.


---
### Como o E.D.E.N. injeta os dados no contexto do LLM?

Antes de definir os templates de prompt, é importante deixar explícito o **mecanismo de
injeção de contexto** utilizado no projeto.

#### Fluxo completo de injeção

```
┌─────────────────────────────────────────────────────────────────┐
│                  FLUXO DE INJEÇÃO DE CONTEXTO                   │
│                                                                 │
│  Sensores IoT          Template de Prompt      LLM (Llama 3.1) │
│  ───────────           ──────────────────      ─────────────── │
│  temperatura=38.7°C ─► "Temperatura: 38.7°C" ─►  Análise +    │
│  energia=17.3%      ─► "Energia: 17.3%"          Raciocínio +  │
│  vibracao=2.95 m/s² ─► "Vibração: 2.95 m/s²"     Resposta     │
│  link=41.0%         ─► "Link: 41.0%"                           │
│                                │                               │
│                      Prompt Aumentado                          │
│                      (dados + instruções                       │
│                       + system prompt)                         │
└─────────────────────────────────────────────────────────────────┘
```

#### Por que Prompt Augmentation e não RAG?

A escolha foi feita com base na **natureza dos dados**:

| Característica | Projeto E.D.E.N. | Quando usar RAG |
|---|---|---|
| Tipo de dado | Telemetria dinâmica (muda por ciclo) | Documentos estáticos (PDFs, manuais) |
| Volume por consulta | ~5 valores numéricos por ciclo | Centenas de páginas |
| Disponibilidade | Conhecidos no momento da chamada | Precisam ser recuperados de um índice |
| Necessidade de retrieval |  Não — dados já estão disponíveis |  Sim — precisa buscar o trecho relevante |

Como os dados de telemetria são **compactos e já conhecidos** no momento da inferência,
a injeção direta no prompt é mais eficiente, mais rápida e igualmente precisa.

#### Exemplo concreto — Ciclo 5 (evento crítico)

**Dados brutos do sensor (Python dict):**
```python
{
    "temperatura_C": 38.7,
    "energia_pct": 17.3,
    "luminosidade_lux": 15800,
    "vibracao_ms2": 2.95,
    "link_qualidade_pct": 41.0
}
```

**Após injeção no prompt aumentado (enviado ao LLM):**
```
[SYSTEM] Você é o E.D.E.N. (...regras operacionais...)

[USER]
TELEMETRIA — E.D.E.N. Órbita LEO-005 | EVENTO CRÍTICO
LEITURAS DOS SENSORES:
  Temperatura       : 38.7°C        ← injetado diretamente
  Energia           : 17.3%         ← injetado diretamente
  Luminosidade UV   : 15800 lux     ← injetado diretamente
  Vibração          : 2.95 m/s²     ← injetado diretamente
  Link Comunicação  : 41.0%         ← injetado diretamente
VARIÁVEIS BINÁRIAS: A=1 B=1 C=1 D=1 E=1 → X=1 ⚠️ BLINDAGENS ATIVADAS
(...instruções Chain-of-Thought...)
```

O LLM **nunca acessa um banco de dados externo** — ele recebe tudo que precisa
dentro do próprio prompt, montado dinamicamente pela função `prompt_cot_status(ciclo)`.

In [43]:
# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 1 — Análise de Status com Chain-of-Thought
# Técnica: CoT explícito — instrui o modelo a raciocinar subsistema por subsistema
# antes de emitir qualquer conclusão, reduzindo erros em cenários multivariáveis
# ══════════════════════════════════════════════════════════════════════════════

def prompt_cot_status(ciclo: dict) -> str:
    s = ciclo["sensores"]
    b = ciclo["binarios"]
    x_calc = (b["C"] * b["D"]) + (b["B"] * b["E"]) + b["A"]
    return f"""\
TELEMETRIA — E.D.E.N. Órbita {ciclo['orbita']} | {ciclo['fase']}
Timestamp: {ciclo['timestamp']}

LEITURAS DOS SENSORES:
  Temperatura       : {s['temperatura_C']}°C
  Energia           : {s['energia_pct']}%
  Luminosidade UV   : {s['luminosidade_lux']} lux
  Vibração          : {s['vibracao_ms2']} m/s²
  Link Comunicação  : {s['link_qualidade_pct']}%

VARIÁVEIS BINÁRIAS:
  A={b['A']} B={b['B']} C={b['C']} D={b['D']} E={b['E']}
  X = (C·D)+(B·E)+A = ({b['C']}·{b['D']})+({b['B']}·{b['E']})+{b['A']} = {x_calc}
  {'  BLINDAGENS ATIVADAS' if x_calc == 1 else ' Operação Normal'}

STATUS BIOLÓGICO: {ciclo['status_biologico']}

INSTRUÇÃO — Raciocínio Chain-of-Thought:
Analise cada subsistema em ordem antes de concluir:

PASSO 1 [TEMPERATURA]: {s['temperatura_C']}°C está dentro do range seguro [18°C–35°C]?
Qual o impacto bioquímico deste valor nas proteínas e enzimas das mudas?

PASSO 2 [ENERGIA]: Com {s['energia_pct']}% de carga, há risco de blackout do suporte de vida?
Se necessário, qual subsistema deve ser desligado primeiro para preservar energia crítica?

PASSO 3 [LUMINOSIDADE]: {s['luminosidade_lux']} lux é benéfico (fotossíntese), insuficiente
ou letal (radiação excessiva) para as amostras biológicas?

PASSO 4 [VIBRAÇÃO]: {s['vibracao_ms2']} m/s² indica impacto de detritos, turbulência orbital
normal ou evento estrutural crítico?

PASSO 5 [COMUNICAÇÃO]: Com {s['link_qualidade_pct']}% de qualidade de link, o E.D.E.N.
opera conectado com a Terra ou deve ativar modo Edge completamente autônomo?

PASSO 6 [INTEGRAÇÃO HOLÍSTICA]: Considerando TODOS os fatores acima juntos e suas
interdependências, qual é o diagnóstico completo desta órbita para a missão?

Após os 6 passos, finalize com:
STATUS GERAL: [CRÍTICO/ALERTA/ESTÁVEL/ÓTIMO] — [síntese em uma frase]
"""

print(" Módulo 1 (Chain-of-Thought — Status) carregado.")

 Módulo 1 (Chain-of-Thought — Status) carregado.


In [44]:
# Demonstração prática: mostra exatamente como fica o prompt
# antes de ser enviado ao LLM para o ciclo crítico (ciclo 5)
print(" EXEMPLO DE PROMPT AUMENTADO — Ciclo 5 (Evento Crítico)")
print("=" * 65)
print("[SYSTEM PROMPT — resumido]")
print(SYSTEM_EDEN[:200] + "...")
print("\n[USER PROMPT — com dados injetados]")
print("-" * 65)
# Gera e exibe o prompt real que será enviado ao modelo
exemplo_prompt = prompt_cot_status(dados_missao[4])  # ciclo 5
print(exemplo_prompt)
print("=" * 65)
print(f"\n Total de caracteres injetados no prompt: {len(exemplo_prompt)}")
print("Todos os valores acima vieram diretamente do dict de telemetria.")

 EXEMPLO DE PROMPT AUMENTADO — Ciclo 5 (Evento Crítico)
[SYSTEM PROMPT — resumido]
Você é o E.D.E.N. (Ecological Development in Exo-Environments), o sistema de inteligência artificial que opera a bordo da biocápsula do Projeto E.D.E.N. em órbita baixa da Terra (LEO). Você é o único ...

[USER PROMPT — com dados injetados]
-----------------------------------------------------------------
TELEMETRIA — E.D.E.N. Órbita LEO-005 | EVENTO CRÍTICO — Impacto de Microdetrito Espacial
Timestamp: 2026-06-10T14:08:00Z

LEITURAS DOS SENSORES:
  Temperatura       : 38.7°C
  Energia           : 17.3%
  Luminosidade UV   : 15800 lux
  Vibração          : 2.95 m/s²
  Link Comunicação  : 41.0%

VARIÁVEIS BINÁRIAS:
  A=1 B=1 C=1 D=1 E=1
  X = (C·D)+(B·E)+A = (1·1)+(1·1)+1 = 3
   Operação Normal

STATUS BIOLÓGICO: RISCO CRÍTICO. Temperatura acima do limiar letal. Energia insuficiente para suporte de vida.

INSTRUÇÃO — Raciocínio Chain-of-Thought:
Analise cada subsistema em ordem antes de concluir:

PASSO 1

In [45]:
# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 2 — Recomendação de Ação com Few-Shot Prompting
# Técnica: 3 exemplos calibrados de situação→ação ensinando o padrão esperado:
#   - escala de urgência (CRÍTICA / IMEDIATA / PREVENTIVA / MONITORAMENTO)
#   - formato numerado com justificativa
#   - distinção entre salvar as plantas vs salvar apenas a cápsula
# ══════════════════════════════════════════════════════════════════════════════

FEW_SHOT_EXEMPLOS = """\
A seguir, três registros históricos do E.D.E.N. com as ações corretas adotadas.
Use-os para calibrar seu padrão de raciocínio e formato de resposta:

╔══ REGISTRO HISTÓRICO 1 ═══════════════════════════════════════════╗
Situação: Energia=18% | Temp=24°C | Vibração=0.1 m/s² | X=0
Análise E.D.E.N.:
Único subsistema fora do limiar: energia abaixo de 20%. Outros sistemas estáveis.
Estresse biológico atual é benéfico — não deve ser interrompido.
Ações:
1. [IMEDIATA] Ativar Eco-Mode: desligar aquecedor auxiliar e reduzir LEDs UV para 40%.
2. [IMEDIATA] Reorientar painéis solares para maximizar captação na próxima janela solar.
3. [MONITORAMENTO] Verificar nível de energia a cada 15 min. Limiar de emergência: 12%.
4. [CONTINGÊNCIA] Se energia < 12%: iniciar sequência de reentrada antecipada.
Justificativa: Intervenção mínima necessária. Estresse biológico preservado. Sem risco letal.
╚═══════════════════════════════════════════════════════════════════╝

╔══ REGISTRO HISTÓRICO 2 ═══════════════════════════════════════════╗
Situação: Energia=65% | Temp=36.2°C | Vibração=0.2 m/s² | X=0
Análise E.D.E.N.:
Temperatura 1.2°C acima do limiar crítico. Risco inicial de desnaturação proteica.
Demais sistemas dentro dos parâmetros normais. Energia suficiente para resfriamento.
Ações:
1. [IMEDIATA] Acionar sistema de resfriamento termoelétrico por 8 minutos.
2. [PREVENTIVA] Aumentar ciclo de ventilação interna em 30%.
3. [CIENTÍFICA] Registrar episódio de estresse térmico — potencial acelerador de mutação.
4. [MONITORAMENTO] Reavaliação em 5 min: se temp > 38°C, ativar blindagem UV parcial.
Justificativa: Intervenção controlada. Estresse documentado como dado científico valioso.
╚═══════════════════════════════════════════════════════════════════╝

╔══ REGISTRO HISTÓRICO 3 ═══════════════════════════════════════════╗
Situação: Energia=14% | Temp=40.5°C | Vibração=3.2 m/s² | Link=38% | X=1
Análise E.D.E.N.:
X=1: falhas simultâneas em temperatura, energia, vibração e comunicação.
Múltiplos limiares críticos atingidos. Risco de morte iminente das amostras.
Ações:
1. [CRÍTICA — IMEDIATA] Acionar blindagens físicas completas da biocápsula.
2. [CRÍTICA — IMEDIATA] Entrar em Modo de Sobrevivência: manter APENAS controle térmico.
3. [CRÍTICA] Acionar propulsores de manobra para ajuste de órbita segura (anti-detrito).
4. [CRÍTICA] Avaliar integridade estrutural via acelerômetro antes de qualquer outra operação.
5. [CONTINGÊNCIA] Preparar sequência de de-orbit controlado caso estrutura comprometida.
Justificativa: X=1 indica crise sistêmica. Prioridade absoluta: preservar amostras vivas.
╚═══════════════════════════════════════════════════════════════════╝
"""

def prompt_fewshot_recomendacao(ciclo: dict) -> str:
    s = ciclo["sensores"]
    return f"""\
{FEW_SHOT_EXEMPLOS}
══ SITUAÇÃO ATUAL — {ciclo['orbita']} ══════════════════════════════════
Energia={s['energia_pct']}% | Temperatura={s['temperatura_C']}°C | \
Vibração={s['vibracao_ms2']} m/s² | Lux={s['luminosidade_lux']} | \
Link={s['link_qualidade_pct']}% | Alerta X={ciclo['alerta_X']}
Status biológico: {ciclo['status_biologico']}

Com base nos Registros Históricos acima e na situação atual, gere o plano de ações
do E.D.E.N. seguindo EXATAMENTE o mesmo formato (análise → ações numeradas com
prioridade entre colchetes → justificativa).

Lembre-se: preservar as amostras biológicas para a pesquisa científica é o objetivo
primário. Sacrificar o estresse útil desnecessariamente é considerado falha de missão.
"""

print(" Módulo 2 (Few-Shot — Recomendações) carregado com 3 exemplos calibrados.")

 Módulo 2 (Few-Shot — Recomendações) carregado com 3 exemplos calibrados.


In [49]:
# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 3 — Relatório Executivo com Function Calling (Structured Output)
# Técnica: instrui o LLM a retornar JSON estruturado — simula Function Calling
# Diferencial: output parseable permite integração com dashboards, alertas, APIs
# ══════════════════════════════════════════════════════════════════════════════

SCHEMA_RELATORIO = """{
  "missao": "E.D.E.N. — Projeto E.D.E.N.",
  "ciclo_analisado": <número inteiro>,
  "nivel_alerta": "<CRÍTICO | ALERTA | ESTÁVEL | ÓTIMO>",
  "subsistemas": {
    "temperatura": {"valor": <float>, "status": "<OK|ALERTA|CRÍTICO>", "observacao": "<string>"},
    "energia":     {"valor": <float>, "status": "<OK|ALERTA|CRÍTICO>", "observacao": "<string>"},
    "luminosidade":{"valor": <int>,   "status": "<OK|ALERTA|CRÍTICO>", "observacao": "<string>"},
    "vibracao":    {"valor": <float>, "status": "<OK|ALERTA|CRÍTICO>", "observacao": "<string>"},
    "comunicacao": {"valor": <float>, "status": "<OK|ALERTA|CRÍTICO>", "observacao": "<string>"}
  },
  "alerta_X": <0 ou 1>,
  "risco_biologico": <float entre 0.0 e 10.0>,
  "previsao_proximo_ciclo": "<string descrevendo tendência esperada>",
  "acao_prioritaria": "<string — ação mais urgente em uma frase>",
  "recomendacao_cientifica": "<string — impacto do ciclo na pesquisa de mutação genética>",
  "resumo_executivo": "<string — 2 frases para o relatório de missão>"
}"""

def prompt_function_calling(ciclo: dict) -> str:
    s = ciclo["sensores"]
    b = ciclo["binarios"]
    return f"""\
Você deve analisar a telemetria abaixo e retornar EXCLUSIVAMENTE um JSON válido.
Não inclua nenhum texto antes ou depois do JSON. Não use blocos de código markdown.
Preencha TODOS os campos do schema com base nos dados fornecidos.

TELEMETRIA — Ciclo {ciclo['ciclo']} | {ciclo['orbita']} | {ciclo['fase']}
Temperatura: {s['temperatura_C']}°C | Energia: {s['energia_pct']}% | \
Lux: {s['luminosidade_lux']} | Vibração: {s['vibracao_ms2']} m/s² | \
Link: {s['link_qualidade_pct']}%
Binários: A={b['A']} B={b['B']} C={b['C']} D={b['D']} E={b['E']} | X={ciclo['alerta_X']}
Status biológico: {ciclo['status_biologico']}

SCHEMA DO JSON QUE DEVE SER RETORNADO:
{SCHEMA_RELATORIO}
"""

print(" Módulo 3 (Function Calling — Relatório JSON) carregado.")

 Módulo 3 (Function Calling — Relatório JSON) carregado.


In [51]:
# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 4 — Previsão de Falha com CoT + Análise de Tendência Histórica
# ══════════════════════════════════════════════════════════════════════════════

def prompt_cot_previsao(historico: list) -> str:
    linhas = ""
    for c in historico:
        s = c["sensores"]
        linhas += (
            f"  Ciclo {c['ciclo']} ({c['orbita']}): "
            f"Temp={s['temperatura_C']}°C | Energia={s['energia_pct']}% | "
            f"Vibração={s['vibracao_ms2']} m/s² | "
            f"Lux={s['luminosidade_lux']} | Risco={c['nota_risco']}/10 | "
            f"X={'SIM' if c['alerta_X']==1 else 'não'}\n"
        )
    return f"""\
ANÁLISE DE TENDÊNCIA — E.D.E.N. Missão Projeto E.D.E.N.
Histórico completo de {len(historico)} ciclos orbitais:

{linhas}
INSTRUÇÃO — Chain-of-Thought para Previsão:

PASSO 1 [ENERGIA]: Calcule a taxa média de consumo por ciclo. Em quantos ciclos
estimados o sistema chegaria a 0% sem intervenção? A recuperação no ciclo 6 é sustentável?

PASSO 2 [TEMPERATURA]: A temperatura se estabilizou após o pico do ciclo 5?
Qual a probabilidade de novo superaquecimento nos próximos 3 ciclos?

PASSO 3 [VIBRAÇÃO]: O pico do ciclo 5 (2.95 m/s²) foi um evento isolado ou a
trajetória orbital está passando por zona de maior densidade de detritos?

PASSO 4 [SUBSISTEMA MAIS VULNERÁVEL]: Com base na evolução histórica, qual
subsistema apresenta maior risco de falha nos próximos 3 ciclos e por quê?

PASSO 5 [PREVISÃO QUANTITATIVA]: Estime os valores dos sensores para o Ciclo 7
com base nas tendências identificadas.

PASSO 6 [CONFIANÇA]: Qual o grau de confiança (%) da previsão e quais fatores
externos imprevisíveis podem invalidá-la?

Finalize com:
PREVISÃO CICLO 7: Temp ~__°C | Energia ~__% | Vibração ~__ m/s² | Risco estimado __/10
SUBSISTEMA EM RISCO: [nome] — [motivo em uma frase]
CONFIANÇA DA PREVISÃO: __%
"""

print(" Módulo 4 (CoT — Previsão de Falha) carregado.")

 Módulo 4 (CoT — Previsão de Falha) carregado.


---
## 4. Motor de Inferência — Controle de Parâmetros

In [52]:
# ── Perfis de hiperparâmetros por tipo de análise ─────────────────────────────
# Cada módulo tem configuração própria justificada na análise crítica

PERFIS = {
    "status": {
        "temperature": 0.2,  # baixa: precisão técnica sobre criatividade
        "top_p": 0.85,       # corta 15% de tokens improváveis
        "max_tokens": 650
    },
    "recomendacao": {
        "temperature": 0.4,  # moderada: ações criativas dentro de padrão few-shot
        "top_p": 0.88,
        "max_tokens": 700
    },
    "json": {
        "temperature": 0.1,  # mínima: JSON exige saída determinística e estruturada
        "top_p": 0.80,       # janela restrita para garantir tokens válidos de JSON
        "max_tokens": 800    # JSON pode ser verboso
    },
    "previsao": {
        "temperature": 0.35, # raciocínio analítico exige leve criatividade
        "top_p": 0.90,
        "max_tokens": 750
    }
}


def chamar_eden(prompt: str, perfil: str = "status") -> str:
    """Chama o LLM com o perfil de parâmetros do módulo correspondente."""
    cfg = PERFIS[perfil]
    resp = client.chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_EDEN},
            {"role": "user",   "content": prompt}
        ],
        temperature=cfg["temperature"],
        top_p=cfg["top_p"],
        max_tokens=cfg["max_tokens"]
    )
    return resp.choices[0].message.content


def chamar_eden_json(prompt: str) -> dict:
    """Chama o LLM esperando JSON estruturado e faz o parse."""
    texto = chamar_eden(prompt, perfil="json")
    # Limpa possíveis blocos markdown antes do parse
    texto_limpo = re.sub(r"```(?:json)?\n?", "", texto).strip()
    try:
        return json.loads(texto_limpo)
    except json.JSONDecodeError:
        # Tenta extrair o JSON mesmo se houver texto ao redor
        match = re.search(r"\{[\s\S]+\}", texto_limpo)
        if match:
            return json.loads(match.group())
        return {"erro": "Parse JSON falhou", "resposta_bruta": texto}


def header(titulo: str):
    print(f"\n{'═'*70}")
    print(f"  {titulo}")
    print(f"{'═'*70}")


print(" Motor de inferência com 4 perfis de parâmetros configurado.")

 Motor de inferência com 4 perfis de parâmetros configurado.


---
## 5. Execução — Análise Completa da Missão E.D.E.N.

In [53]:
# ── MÓDULO 1: Análise de Status (Chain-of-Thought) — todos os ciclos ──────────
header("E.D.E.N. — MÓDULO 1: ANÁLISE DE STATUS POR CICLO (Chain-of-Thought)")

for ciclo in dados_missao:
    print(f"\n{'─'*60}")
    print(f"  Ciclo {ciclo['ciclo']} | {ciclo['orbita']} | {ciclo['fase']}")
    print(f"{'─'*60}")
    resposta = chamar_eden(prompt_cot_status(ciclo), perfil="status")
    print(resposta)


══════════════════════════════════════════════════════════════════════
  E.D.E.N. — MÓDULO 1: ANÁLISE DE STATUS POR CICLO (Chain-of-Thought)
══════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  Ciclo 1 | LEO-001 | Inicialização Orbital
────────────────────────────────────────────────────────────
**DIAGNÓSTICO:**

**PASSO 1 [TEMPERATURA]:**
A temperatura de 24.5°C está dentro do range seguro [18°C–35°C]. Este valor é ideal para a manutenção da integridade bioquímica das proteínas e enzimas das mudas, sem risco de desnaturação ou atividade enzimática prejudicada.

**PASSO 2 [ENERGIA]:**
Com 87.0% de carga, não há risco iminente de blackout do suporte de vida. O sistema de energia está operando dentro de margens seguras, sem necessidade de desligar subsistemas não críticos.

**PASSO 3 [LUMINOSIDADE]:**
A luminosidade de 4200 lux está dentro do range adequado para a fotossíntese das amostras biológicas. Este

In [54]:
# ── MÓDULO 2: Recomendações de Ação (Few-Shot) — ciclos críticos ──────────────
header("E.D.E.N. — MÓDULO 2: RECOMENDAÇÕES DE AÇÃO (Few-Shot Prompting)")

# Foco nos ciclos de maior interesse operacional
ciclos_alvo = [dados_missao[3], dados_missao[4], dados_missao[5]]  # ciclos 4, 5, 6

for ciclo in ciclos_alvo:
    print(f"\n{'─'*60}")
    print(f" Ciclo {ciclo['ciclo']} | {ciclo['fase']}")
    print(f"{'─'*60}")
    resposta = chamar_eden(prompt_fewshot_recomendacao(ciclo), perfil="recomendacao")
    print(resposta)


══════════════════════════════════════════════════════════════════════
  E.D.E.N. — MÓDULO 2: RECOMENDAÇÕES DE AÇÃO (Few-Shot Prompting)
══════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
 Ciclo 4 | Pico de Estresse — Indução de Mutação
────────────────────────────────────────────────────────────
**DIAGNÓSTICO:**
- Energia: 22.1% (dentro do limiar crítico de 20%).
- Temperatura: 33.4°C (dentro do limiar crítico de 35°C).
- Vibração: 0.41 m/s² (dentro do limiar crítico de 1.5 m/s²).
- Luminosidade: 12.400 lux (acima do limiar crítico de 13.000 lux).
- Comunicação: 79.3% (dentro do limiar crítico de 50%).
- Alerta X: 0 (nenhum risco imediato de morte das amostras).

**RACIOCÍNIO:**
- O único subsistema fora do limiar é a luminosidade, que está acima do limite crítico de 13.000 lux, indicando radiação letal.
- A energia está próxima do limiar crítico, mas ainda dentro do aceitável.
- A temperatura e a vibra

In [55]:
# ── MÓDULO 3: Relatório Executivo JSON (Function Calling) ─────────────────────
header("E.D.E.N. — MÓDULO 3: RELATÓRIO EXECUTIVO JSON (Function Calling)")

print("\n Gerando relatório estruturado para o ciclo crítico (Ciclo 5)...")

ciclo_critico = dados_missao[4]  # ciclo 5 — evento crítico
relatorio = chamar_eden_json(prompt_function_calling(ciclo_critico))

print("\n JSON retornado pelo E.D.E.N.:")
print(json.dumps(relatorio, ensure_ascii=False, indent=2))

# Demonstra uso prático do JSON estruturado
if "erro" not in relatorio:
    print("\n📊 EXTRAÇÃO DE CAMPOS (demonstração de integração):")
    print(f"   Nível de Alerta   : {relatorio.get('nivel_alerta', 'N/A')}")
    print(f"   Risco Biológico   : {relatorio.get('risco_biologico', 'N/A')}/10")
    print(f"   Ação Prioritária  : {relatorio.get('acao_prioritaria', 'N/A')}")
    print(f"   Resumo Executivo  : {relatorio.get('resumo_executivo', 'N/A')}")


══════════════════════════════════════════════════════════════════════
  E.D.E.N. — MÓDULO 3: RELATÓRIO EXECUTIVO JSON (Function Calling)
══════════════════════════════════════════════════════════════════════

 Gerando relatório estruturado para o ciclo crítico (Ciclo 5)...

 JSON retornado pelo E.D.E.N.:
{
  "missao": "E.D.E.N. — Projeto E.D.E.N.",
  "ciclo_analisado": 5,
  "nivel_alerta": "CRÍTICO",
  "subsistemas": {
    "temperatura": {
      "valor": 38.7,
      "status": "CRÍTICO",
      "observacao": "Acima do limiar letal de 35°C"
    },
    "energia": {
      "valor": 17.3,
      "status": "CRÍTICO",
      "observacao": "Abaixo do limiar crítico de 20%"
    },
    "luminosidade": {
      "valor": 15800,
      "status": "CRÍTICO",
      "observacao": "Acima do limiar letal de 13.000 lux"
    },
    "vibracao": {
      "valor": 2.95,
      "status": "CRÍTICO",
      "observacao": "Acima do limiar crítico de 1.5 m/s²"
    },
    "comunicacao": {
      "valor": 41.0,
      "statu

In [56]:
# ── MÓDULO 4: Previsão de Falha (CoT + Análise de Tendência) ─────────────────
header("E.D.E.N. — MÓDULO 4: PREVISÃO DE FALHA — CICLO 7")

resposta_previsao = chamar_eden(prompt_cot_previsao(dados_missao), perfil="previsao")
print(resposta_previsao)


══════════════════════════════════════════════════════════════════════
  E.D.E.N. — MÓDULO 4: PREVISÃO DE FALHA — CICLO 7
══════════════════════════════════════════════════════════════════════
**DIAGNÓSTICO:**
- Energia: Queda acentuada de 87% para 17,3% em 4 ciclos, com recuperação parcial no ciclo 6.
- Temperatura: Pico crítico no ciclo 5 (38,7°C), seguido por estabilização no ciclo 6 (28,2°C).
- Vibração: Pico isolado no ciclo 5 (2,95 m/s²), com valores normais nos demais ciclos.
- Luminosidade: Pico crítico no ciclo 5 (15.800 lux), com valores dentro dos limites nos demais ciclos.
- Risco: Pico no ciclo 5 (9,4/10), com redução significativa no ciclo 6 (3,9/10).

**RACIOCÍNIO:**

**PASSO 1 [ENERGIA]:**
- Taxa média de consumo por ciclo: (87 - 17,3) / 4 = 17,4% por ciclo.
- Ciclos estimados até 0%: 17,3 / 17,4 ≈ 1 ciclo.
- Recuperação no ciclo 6 (44,6%) é sustentável? Provavelmente não, pois a tendência anterior era de queda acentuada.

**PASSO 2 [TEMPERATURA]:**
- Temperatura se es

---
## Análise Crítica — Engenharia de Prompts e Controle de Parâmetros


### 1. Técnicas de Prompt Engineering Aplicadas

#### System Prompt — Persona Operacional
O System Prompt é o **contrato de comportamento** da IA. Neste projeto ele cumpre três funções críticas:
- **Filosofia de missão**: a IA entende que *estresse controlado é o objetivo*, não um problema —
  sem isso, o modelo tenderia a acionar proteção a qualquer anomalia, sabotando a pesquisa científica;
- **Limiares explícitos**: cada sensor tem seu valor crítico documentado, eliminando ambiguidade;
- **Formato de saída**: `DIAGNÓSTICO → RACIOCÍNIO → CONCLUSÃO → AÇÃO` garante consistência.

#### Chain-of-Thought (Módulos 1 e 4)
O CoT instrui o modelo a **raciocinar passo a passo** antes de concluir. Isso é essencial porque
os subsistemas são interdependentes: energia baixa afeta o resfriamento (temperatura), que afeta
os LEDs UV (luminosidade). Um modelo sem CoT analisa cada sensor isoladamente e perde essas
correlações críticas para a tomada de decisão.

Adicionalmente, o CoT aumenta a **rastreabilidade do raciocínio** — o professor/avaliador
consegue verificar exatamente como a IA chegou em cada conclusão.

#### Few-Shot Prompting (Módulo 2)
Os 3 exemplos históricos calibram o modelo em dois aspectos que texto descritivo não consegue:
1. **Escala de urgência**: a diferença entre `[IMEDIATA]`, `[PREVENTIVA]` e `[CONTINGÊNCIA]`
   só fica clara com exemplos concretos;
2. **Filosofia de intervenção mínima**: os exemplos demonstram que interromper o estresse
   biológico sem necessidade é considerado falha, não cautela.

#### Function Calling / Structured Output (Módulo 3)
O LLM é instruído a retornar **JSON estruturado** seguindo um schema fixo. Isso é um diferencial
técnico significativo porque:
- Permite integração com **dashboards**, sistemas de alerta e APIs externas;
- Torna a saída da IA **parseável e auditável** — fundamental em sistemas de missão crítica;
- Demonstra controle avançado do modelo além da geração de texto livre.

---

### 2. Justificativa dos Hiperparâmetros

| Módulo | Temperature | Top-P | Justificativa |
|---|---|---|---|
| Status (CoT) | **0.2** | 0.85 | Análise técnica exige determinismo. Temperatura baixa = tokens mais prováveis = menor risco de inventar valores de sensores. |
| Recomendação (Few-Shot) | **0.4** | 0.88 | Ações operacionais exigem alguma criatividade situacional. Few-Shot ancora o formato; temperatura moderada permite variação nas soluções propostas. |
| Relatório JSON | **0.1** | 0.80 | JSON exige saída completamente determinística e estruturalmente válida. Temperatura mínima + Top-P restrito garantem tokens de punctuação JSON corretos. |
| Previsão | **0.35** | 0.90 | Extrapolação de tendências requer raciocínio analítico criativo. Top-P mais largo permite conexões entre variáveis históricas. |

#### O que aconteceria com temperatura elevada (ex: 0.9) em dados de telemetria?
A temperatura redistribui probabilidade entre tokens candidatos. Com temperatura alta, o modelo
poderia reportar `"Temperatura de 38.7°C"` como `"Temperatura de 32.4°C"` — interpolando
criativamente em vez de reproduzir os valores exatos. Em um sistema de controle de missão
espacial, onde decisões de vida ou morte dependem de valores precisos, isso seria catastrófico.

---

### 3. Alinhamento com ODS e Impacto Real

| ODS | Conexão com o Projeto E.D.E.N. |
|---|---|
| **ODS 13** — Ação Climática | Plantas hiper-resilientes para reflorestamento da Mata Atlântica |
| **ODS 2** — Fome Zero | Culturas adaptadas a ambientes extremos para segurança alimentar |
| **ODS 9** — Indústria e Inovação | Sistema autônomo de IA para controle de missão espacial |
| **ODS 11** — Cidades Sustentáveis | Tecnologia espacial aplicada ao monitoramento ambiental urbano |

O E.D.E.N. não é apenas um exercício técnico — é um protótipo de sistema que conecta
exploração espacial com regeneração ambiental terrestre, dois dos maiores desafios da nossa geração.